In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

print(os.getcwd())

meta_df = pd.read_parquet(
     "003_fnr_value_learning_extended/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
C_vals = meta_df["C_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[8] # 0.888
eta = eta_vals[1] # 0.111
gamma = 0
C = 1
alpha = 1
alpha2 = alpha_vals[4]
kappa = 1
kappa2 = kappa_vals[2]

print(lambda_A, eta, gamma, C, alpha, kappa)

filters = [
    ("lambda_A", "=", lambda_A),
    ("eta", "=", eta),
    ("gamma", "=", gamma),
    ("C", "=", C),
    ("alpha", "=", alpha),
    ("kappa", "=", kappa)
]

# read only filtered rows and needed columns
cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std"]

table = pq.read_table(
        "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_summary.parquet",
    columns=cols_needed,
    filters=filters
)

dff = table.to_pandas().sort_values("Step")

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c", "d"),
)

# --- V (top) ---
fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([V_upper, V_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(0,0,255,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)

# --- M_A (bottom) ---
fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.add_hline(y=0, row=2, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.4, 0.4],
    row=2,
    col=1,
)

# Alpha 0.444

filters = [
    ("lambda_A", "=", lambda_A),
    ("eta", "=", eta),
    ("gamma", "=", gamma),
    ("C", "=", C),
    ("alpha", "=", alpha2),
    ("kappa", "=", kappa)
]

# read only filtered rows and needed columns
cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std"]

table = pq.read_table(
        "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_summary.parquet",
    columns=cols_needed,
    filters=filters
)

dff = table.to_pandas().sort_values("Step")

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_hline(y=0, row=3, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.4, 0.4],
    row=3,
    col=1,
)

# Kappa = 1.889

filters = [
    ("lambda_A", "=", lambda_A),
    ("eta", "=", eta),
    ("gamma", "=", gamma),
    ("C", "=", C),
    ("alpha", "=", alpha2),
    ("kappa", "=", kappa2)
]

# read only filtered rows and needed columns
cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std"]

table = pq.read_table(
        "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_summary.parquet",
    columns=cols_needed,
    filters=filters
)

dff = table.to_pandas().sort_values("Step")

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=3, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.4, 0.4],
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_xaxes(title_text="Step", row=4, col=1)

# ################

fig.update_annotations(xshift=-360, yshift=-20)
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=50),
    height=900,
    showlegend=False,
)

fig.write_image("plots/plot1.pdf")

/home/soelderer/projects/formal_model_irritability/simulations
0.8888889 0.11111111 0 1 1 1
